# LangChain: Q&A over Documents

Load a CSV product catalog, embed it into an in-memory vector store,
and answer natural-language questions about it using retrieval-augmented generation (RAG).

**Concept flow:**
```
CSV file
   │  CSVLoader
   ▼
Documents  ──  OpenAIEmbeddings  ──►  DocArrayInMemorySearch (vector store)
                                              │
                                         retriever
                                              │
                              question ──► RAG chain ──► answer
```

## Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

llm_model = 'gpt-3.5-turbo'

In [ ]:
# Install if missing:
# pip install langchain-openai langchain-community docarray

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import CSVLoader
from langchain_community.vectorstores import DocArrayInMemorySearch
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from IPython.display import display, Markdown

## Step 1 — Load documents from CSV

`CSVLoader` reads each CSV row as a separate `Document` object.
Each document's `page_content` is the row serialized as `column: value` text — this is what gets embedded and searched later.

**Problem with raw loading:** the CSV has an unnamed index column (row numbers 0, 1, 2…)
which shows up as `: 0`, `: 1` noise in the text. We fix this by reading the CSV
with pandas first to identify and drop that column before loading.

In [ ]:
import pandas as pd

# Inspect the raw CSV to see column names
df = pd.read_csv('assets/OutdoorClothingCatalog_1000.csv')
print('Columns:', df.columns.tolist())
print(f'Rows   : {len(df)}')
print()
print(df.head(2))

In [ ]:
# The first column is an unnamed row index — drop it, save a clean CSV
df_clean = df.drop(columns=df.columns[df.columns.str.startswith('Unnamed')], errors='ignore')
clean_path = 'assets/OutdoorClothingCatalog_clean.csv'
df_clean.to_csv(clean_path, index=False)

# Load the clean CSV — each row becomes one Document
loader = CSVLoader(file_path=clean_path)
docs = loader.load()

print(f'Loaded {len(docs)} documents')
print('\nFirst document page_content:')
print(docs[0].page_content[:300])

## Step 2 — Embed documents into a vector store

`OpenAIEmbeddings` converts any text into a list of 1536 numbers (a *vector*).
The key property: **similar meaning → similar numbers**.
So `'UV protection shirt'` and `'sunblock top'` will have vectors very close to each other,
even though they share no words.

`DocArrayInMemorySearch` stores all those vectors in RAM.
When you search, it computes which stored vectors are closest to your query vector
and returns those documents — this is called *semantic search*.

In [ ]:
embeddings = OpenAIEmbeddings()

# Sanity check: embed one sentence and inspect the vector
sample = embeddings.embed_query('sun protection shirt')
print(f'Vector dimensions : {len(sample)}')
print(f'First 5 values    : {[round(v, 4) for v in sample[:5]]}')
print('(Each number encodes some aspect of the meaning of the text)')

In [ ]:
# Embed all 1000 documents and store them — this makes one API call per batch
vectorstore = DocArrayInMemorySearch.from_documents(docs, embeddings)
print('Vector store built.')

## Step 3 — Test similarity search

Before building the full RAG chain, let's verify the retriever works correctly.

A `retriever` wraps the vector store. When you call `retriever.invoke(query)` it:
1. Embeds your query into a vector
2. Finds the `k` stored documents whose vectors are most similar
3. Returns those documents

**Why k=8 for the sun protection question?**
The question asks to *list all* shirts — we need enough results to cover all variants.
For a targeted question like 'suggest one jacket' k=4 is fine.

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={'k': 8})

test_query = 'shirt with sun protection'
retrieved = retriever.invoke(test_query)

print(f'Retrieved {len(retrieved)} documents for: "{test_query}"')
print()
for i, doc in enumerate(retrieved):
    # Show just the first line (product name) of each result
    first_line = doc.page_content.split('\n')[0]
    print(f'  {i+1}. {first_line}')

## Step 4 — Build the RAG chain

RAG = **R**etrieval-**A**ugmented **G**eneration.

The idea: instead of asking the LLM to answer from memory (which may be wrong or outdated),
we first fetch the relevant product descriptions from our catalog,
then hand them to the LLM as context and say *'answer based only on this'*.

The LCEL pipeline step by step:
```
question
   │
   ├──► retriever       # fetch 8 relevant docs from vector store
   │        │
   │    format_docs     # join doc texts into one context string
   │        │
   └──► RunnablePassthrough  # question passes through unchanged
            │
        prompt          # fills {context} and {question} into the template
            │
           llm          # generates the answer
            │
       StrOutputParser  # extracts plain string from AIMessage
            │
          answer
```

In [ ]:
llm = ChatOpenAI(temperature=0.0, model=llm_model)

prompt = ChatPromptTemplate.from_template(
    'You are a helpful shopping assistant for an outdoor clothing store.\n'
    'Answer the question based ONLY on the product descriptions below.\n'
    'If the answer is not in the context, say "I don\'t have that information."\n\n'
    'PRODUCT CATALOG:\n{context}\n\n'
    'QUESTION: {question}'
)

def format_docs(documents):
    """Join retrieved document texts into one context string for the prompt."""
    return '\n\n---\n\n'.join(doc.page_content for doc in documents)

rag_chain = (
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

## Step 5 — Ask questions

The chain retrieves relevant products from the catalog and answers based on them.
It cannot hallucinate products that aren't in the CSV.

In [ ]:
question = (
    'Please list all your shirts with sun protection '
    'in a table in markdown and summarize each one.'
)

response = rag_chain.invoke(question)
display(Markdown(response))

In [ ]:
response = rag_chain.invoke('Do you have any waterproof jackets?')
display(Markdown(response))

In [ ]:
response = rag_chain.invoke('What is the most affordable option for hiking?')
display(Markdown(response))

## Bonus — One-liner with `VectorstoreIndexCreator`

`VectorstoreIndexCreator` wraps Steps 1–3 (load → embed → store) into one call.
Good for quick experiments. The step-by-step approach above gives you full control
over retriever settings, prompts, and the chain structure.

In [ ]:
from langchain_community.indexes import VectorstoreIndexCreator

index = VectorstoreIndexCreator(
    vectorstore_cls=DocArrayInMemorySearch,
    embedding=OpenAIEmbeddings(),
).from_loaders([CSVLoader(file_path=clean_path)])

response = index.query(question, llm=ChatOpenAI(temperature=0, model=llm_model))
display(Markdown(response))